# 甲状腺治疗结果预测数据说明

## 数据概述
- **样本总数**: 700例患者
- **数据来源**: `700.xlsx` (甲状腺放射性碘治疗记录)
- **预测目标**: 治疗后结果 (三分类)
  - **类别1 (甲亢)**: 225例 (32.1%)
  - **类别2 (甲减)**: 168例 (24.0%)
  - **类别3 (正常/治愈)**: 307例 (43.9%)

## 特征变量
### 基础特征
- 年龄、体重、甲状腺重量、超声血流评分
- 激素指标: FT3, FT4, TSH
- 抗体指标: TRAb, TGAb, TPOAb
- 治疗参数: 碘-131剂量(mCi)、24h吸碘率

### 工程特征
- **剂量强度**: 剂量 ÷ 甲状腺重量 (mCi/g)
- **抗体密度**: TRAb ÷ 甲状腺重量
- **有效剂量**: 剂量 × 吸碘率 (实际吸收的辐射量)

## 数据质量
- 存在缺失值，使用中位数填补
- 超声文本需转换为0-3数值评分
- 样本量适中但类别略不平衡


# Random Forest (随机森林) 算法说明

## 算法原理
随机森林是一种**集成学习**方法，通过构建多棵决策树并综合它们的预测结果来提高准确率。

### 核心机制
1. **Bootstrap采样**: 从原始数据中有放回地随机抽样，构建多个训练子集
2. **随机特征选择**: 每次分裂节点时随机选择部分特征，降低树之间的相关性
3. **多数投票**: 对于分类问题，最终预测由所有树的投票决定

## 模型配置
- **交叉验证**: 5折分层交叉验证 (确保每折类别比例一致)
- **类别平衡**: `class_weight='balanced'` 处理样本不平衡问题
- **自动调优**: RandomizedSearchCV搜索最佳超参数
  - 树的数量 (n_estimators)
  - 最大深度 (max_depth)
  - 最小分裂样本数 (min_samples_split)
  - 特征选择策略 (max_features)

## 本数据集结果
- **最佳准确率**: 50% (经过自动调优)
- **Top特征重要性**:
  1. 甲状腺重量 (20.9%)
  2. 有效剂量 (13.9%)
  3. 剂量强度 (12.3%)

## 结论
⚠️ **RF在此数据集上表现不佳** (仅50%准确率，远低于预期)，说明：
- 可能存在数据质量问题
- 特征与结果的关系可能高度非线性
- 需要尝试更先进的模型 (如TabPFN、XGBoost等)


In [5]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict, RandomizedSearchCV
from sklearn.metrics import classification_report, accuracy_score
from sklearn.impute import SimpleImputer

# ============================
# 1. 数据加载与清洗 (通用版)
# ============================
def load_data(file_path):
    if file_path.endswith('.xlsx'):
        df = pd.read_excel(file_path, header=None, engine='openpyxl')
    else:
        df = pd.read_csv(file_path, header=None, encoding='utf-8')

    data = df.iloc[2:].copy()
    
    # 映射列名
    column_map = {
        0: 'RandomID', 4: 'Age', 6: 'Weight', 9: 'Thyroid_Weight', 
        10: 'Ultrasound_Text', 14: 'Outcome', 
        16: 'FT3', 17: 'FT4', 18: 'TSH', 21: 'TRAb',
        22: 'Iodine_Uptake_24h', 25: 'Dose_mCi'
    }
    data = data.rename(columns=column_map)[list(column_map.values())]
    
    # 清洗数值列
    cols = ['Age', 'Weight', 'Thyroid_Weight', 'FT3', 'FT4', 'TSH', 'TRAb', 'Iodine_Uptake_24h', 'Dose_mCi', 'Outcome']
    for c in cols:
        data[c] = pd.to_numeric(data[c], errors='coerce')
    
    data = data.dropna(subset=['Outcome']) # 删掉没结局的
    
    # 处理超声文本
    def map_us(t):
        t = str(t)
        if '极' in t or '异常' in t: return 3
        if '增多' in t or '丰富' in t: return 2
        if '未见' in t: return 0
        return 1
    data['Ultrasound_Score'] = data['Ultrasound_Text'].apply(map_us)
    
    return data

# ============================
# 2. 强力特征工程
# ============================
def get_features(df):
    # 1. 剂量强度: 每克甲状腺给了多少药?
    df['Dose_Intensity'] = df['Dose_mCi'] / df['Thyroid_Weight']
    # 2. 抗体密度: 每克甲状腺有多少抗体?
    df['TRAb_Density'] = df['TRAb'] / df['Thyroid_Weight']
    # 3. 吸收量估计: 剂量 * 吸碘率 (实际吃到肚子里的辐射)
    df['Effective_Dose'] = df['Dose_mCi'] * (df['Iodine_Uptake_24h'] / 100)
    
    features = ['Age', 'Weight', 'Thyroid_Weight', 'Ultrasound_Score', 
                'FT3', 'TRAb', 'Iodine_Uptake_24h', 'Dose_mCi',
                'Dose_Intensity', 'TRAb_Density', 'Effective_Dose']
    
    X = df[features]
    # 填补缺失值 (用中位数)
    X = X.fillna(X.median())
    
    # --- 改为三分类：保持原始Outcome (1,2,3) ---
    y = df['Outcome'].astype(int)
    
    return X, y, features

# ============================
# 3. 自动调优函数
# ============================
def auto_tune_rf(X, y):
    """使用RandomizedSearchCV自动寻找最佳参数"""
    print("\n🔧 开始自动调优 (这可能需要几分钟)...")
    
    # 定义参数搜索空间
    param_distributions = {
        'n_estimators': [100, 200, 300, 500],
        'max_depth': [3, 5, 7, 10, 15, None],
        'min_samples_split': [2, 5, 10, 20],
        'min_samples_leaf': [1, 2, 4, 8],
        'max_features': ['sqrt', 'log2', 0.3, 0.5],
        'class_weight': ['balanced', 'balanced_subsample']
    }
    
    # 基础模型
    rf_base = RandomForestClassifier(random_state=42)
    
    # 随机搜索 (尝试30种组合)
    random_search = RandomizedSearchCV(
        estimator=rf_base,
        param_distributions=param_distributions,
        n_iter=30,  # 尝试30种参数组合
        cv=5,       # 5折交叉验证
        scoring='accuracy',
        n_jobs=-1,  # 使用所有CPU核心
        random_state=42,
        verbose=1
    )
    
    random_search.fit(X, y)
    
    print("\n✅ 调优完成！")
    print(f"最佳准确率: {random_search.best_score_:.4f}")
    print(f"最佳参数: {random_search.best_params_}")
    
    return random_search.best_estimator_

# ============================
# 4. 运行随机森林
# ============================
def run_rf_model(file_path, auto_tune=False):
    print("正在加载数据...")
    df = load_data(file_path)
    X, y, feat_names = get_features(df)
    
    print(f"样本总数: {len(X)}")
    print(f"类别1(甲亢)数量: {sum(y==1)}")
    print(f"类别2(甲减)数量: {sum(y==2)}")
    print(f"类别3(正常)数量: {sum(y==3)}")
    
    if auto_tune:
        # 自动调优模式
        rf = auto_tune_rf(X, y)
    else:
        # 默认参数
        print("\n使用默认参数 (如需自动调优，设置 auto_tune=True)")
        rf = RandomForestClassifier(
            n_estimators=200, 
            max_depth=5, 
            class_weight='balanced', 
            random_state=42
        )
    
    # 使用 5折交叉验证 (Cross Validation)
    print("\n正在进行 5折交叉验证...")
    y_pred = cross_val_predict(rf, X, y, cv=5)
    
    print("\n--- 最终成绩单 ---")
    print(classification_report(y, y_pred, target_names=['甲亢(1)', '甲减(2)', '正常(3)']))
    
    # 查看最重要的特征
    rf.fit(X, y) # 在全量数据上再拟合一次看特征重要性
    importances = pd.DataFrame({'Feature': feat_names, 'Importance': rf.feature_importances_})
    importances = importances.sort_values(by='Importance', ascending=False)
    
    print("\n--- 决定成败的关键因素 (Top 5) ---")
    print(importances.head(5).to_string(index=False))

if __name__ == "__main__":
    file_name = "700.xlsx" # 改成你的文件名
    
    # ⚠️ 设置 auto_tune=True 启用自动调优 (需要几分钟)
    # ⚠️ 设置 auto_tune=False 使用默认参数 (快速)
    USE_AUTO_TUNE = True  # 👈 改这里
    
    try:
        run_rf_model(file_name, auto_tune=USE_AUTO_TUNE)
    except Exception as e:
        print(f"Error: {e}")


正在加载数据...
样本总数: 700
类别1(甲亢)数量: 225
类别2(甲减)数量: 168
类别3(正常)数量: 307

🔧 开始自动调优 (这可能需要几分钟)...
Fitting 5 folds for each of 30 candidates, totalling 150 fits

✅ 调优完成！
最佳准确率: 0.5000
最佳参数: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': 15, 'class_weight': 'balanced'}

正在进行 5折交叉验证...

--- 最终成绩单 ---
              precision    recall  f1-score   support

       甲亢(1)       0.52      0.44      0.47       225
       甲减(2)       0.42      0.28      0.34       168
       正常(3)       0.52      0.67      0.58       307

    accuracy                           0.50       700
   macro avg       0.48      0.46      0.46       700
weighted avg       0.49      0.50      0.49       700


--- 决定成败的关键因素 (Top 5) ---
       Feature  Importance
Thyroid_Weight    0.135869
Dose_Intensity    0.118925
Effective_Dose    0.111350
  TRAb_Density    0.100535
          TRAb    0.099431


# TabPFN (Tabular Prior-Data Fitted Network) 说明

## 算法原理
TabPFN是一个**预训练的Transformer分类器**，专门为小规模表格数据设计（<1000样本）。

### 核心创新：In-Context Learning（上下文学习）
- **类似GPT的工作方式**：不需要训练参数，直接从"上下文"（训练数据）中学习
- **预训练策略**：在数百万个合成表格数据集上预训练，学会"如何做分类"这个元任务
- **推理过程**：
  1. 将训练集和测试集作为输入序列喂给Transformer
  2. 模型根据训练样本的模式直接预测测试样本
  3. 整个过程无需更新模型权重

### 与传统机器学习的区别
| 特性 | 传统ML (如RF) | TabPFN |
|------|--------------|--------|
| 训练方式 | 每次都重新训练 | 预训练模型，直接推理 |
| 适用数据量 | 无限制 | <1000样本 |
| 速度 | 中等 | 较慢（Transformer推理） |
| 优势场景 | 大数据、简单模式 | 小数据、复杂非线性 |

## 本数据集结果
- **准确率**: 47.8% (比RF的50%还差)
- **问题分析**:
  - 类别2(甲减)几乎没预测对 (recall=0.03)
  - 模型极度偏向预测类别3
  - 可能原因：数据噪音太大 或 特征-结果关系过于随机

## 结论
⚠️ **TabPFN也未能有效学习此数据的模式**，说明：
- 数据质量可能存在根本性问题
- 治疗结果可能受未记录的混杂因素影响
- 建议：数据清洗、增加特征、或重新审视问题定义


In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.impute import SimpleImputer

# ==========================================
# 1. 数据加载函数（简化版，与RF共用）
# ==========================================
def load_and_clean_data(file_path):
    if file_path.endswith('.xlsx') or file_path.endswith('.xls'):
        df_raw = pd.read_excel(file_path, header=None, engine='openpyxl')
    else:
        try:
            df_raw = pd.read_csv(file_path, header=None, encoding='utf-8')
        except UnicodeDecodeError:
            df_raw = pd.read_csv(file_path, header=None, encoding='gbk')
            
    data = df_raw.iloc[2:].copy()
    
    column_map = {
        0: 'RandomID', 4: 'Age', 6: 'Weight', 9: 'Thyroid_Weight', 
        10: 'Ultrasound_Text', 14: 'Outcome', 
        16: 'FT3', 17: 'FT4', 18: 'TSH', 21: 'TRAb',
        22: 'Iodine_Uptake_24h', 25: 'Dose_mCi'
    }
    data = data.rename(columns=column_map)
    data = data[list(column_map.values())] 
    
    # 处理超声文本
    def map_ultrasound(text):
        if pd.isna(text): return np.nan
        text = str(text)
        if any(x in text for x in ['极', '异常', '丰富']): return 3
        if any(x in text for x in ['增多']): return 2
        if '未见' in text: return 0
        return 1 
    
    data['Ultrasound_Score'] = data['Ultrasound_Text'].apply(map_ultrasound)
    
    # 数值列清洗
    numeric_cols = ['Age', 'Weight', 'Thyroid_Weight', 'FT3', 'FT4', 'TSH', 'TRAb', 
                    'Iodine_Uptake_24h', 'Dose_mCi', 'Outcome']
    
    for col in numeric_cols:
        data[col] = pd.to_numeric(data[col], errors='coerce')

    data = data.dropna(subset=['Outcome'])
    
    return data

# ==========================================
# 2. 特征工程（与RF一致）
# ==========================================
def engineer_features(df):
    df['Dose_Intensity'] = df['Dose_mCi'] / df['Thyroid_Weight']
    df['TRAb_Density'] = df['TRAb'] / df['Thyroid_Weight']
    df['Effective_Dose'] = df['Dose_mCi'] * (df['Iodine_Uptake_24h'] / 100)
    
    feature_cols = [
        'Age', 'Weight', 'Thyroid_Weight', 'Ultrasound_Score', 
        'FT3', 'TRAb', 'Iodine_Uptake_24h', 'Dose_mCi',
        'Dose_Intensity', 'TRAb_Density', 'Effective_Dose'
    ]
    
    X = df[feature_cols]
    y = df['Outcome'].astype(int)
    
    return X, y, feature_cols

# ==========================================
# 3. TabPFN模型
# ==========================================
def run_tabpfn_pipeline(file_path):
    print("正在加载数据...")
    df = load_and_clean_data(file_path)
    X, y, feat_names = engineer_features(df)
    
    print(f"样本总数: {len(X)}")
    print(f"类别1(甲亢)数量: {sum(y==1)}")
    print(f"类别2(甲减)数量: {sum(y==2)}")
    print(f"类别3(正常)数量: {sum(y==3)}")
    
    # 填补缺失值
    imputer = SimpleImputer(strategy='median')
    X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)
    
    # 切分数据（20%测试集）
    X_train, X_test, y_train, y_test = train_test_split(
        X_imputed, y, test_size=0.2, random_state=42, stratify=y
    )
    
    print(f"\n训练集: {len(X_train)}, 测试集: {len(X_test)}")
    
    try:
        from tabpfn import TabPFNClassifier
    except ImportError as e:
        print(f"❌ TabPFN导入失败: {e}")
        print("请运行: pip install tabpfn")
        raise

    # 初始化TabPFN（预训练模型，无需训练）
    classifier = TabPFNClassifier(device='cpu') 
    
    print("正在进行 In-Context Learning...")
    classifier.fit(X_train, y_train)  # 这里的"fit"只是记录上下文，不训练参数
    y_pred = classifier.predict(X_test)
    
    # 评估
    print("\n--- 最终成绩单 ---")
    print(f"准确率: {accuracy_score(y_test, y_pred):.4f}")
    print("\n详细分类报告:")
    print(classification_report(y_test, y_pred, target_names=['甲亢(1)', '甲减(2)', '正常(3)']))
    
    return classifier, X_test, y_test

# ==========================================
# 主程序
# ==========================================
if __name__ == "__main__":
    file_name = "700.xlsx"
    
    try:
        model, X_test, y_test = run_tabpfn_pipeline(file_name)
        print("\n✅ TabPFN运行完成！")
    except Exception as e:
        print(f"\n❌ 运行出错: {e}")


正在加载数据...
样本总数: 700
类别1(甲亢)数量: 225
类别2(甲减)数量: 168
类别3(正常)数量: 307

训练集: 560, 测试集: 140
正在进行 In-Context Learning...


/home/l1q/anaconda3/envs/med/lib/python3.10/site-packages/tabpfn/classifier.py:604: UserWarning: Running on CPU with more than 200 samples may be slow.
Consider using a GPU or the tabpfn-client API: https://github.com/PriorLabs/tabpfn-client
  check_cpu_warning(



--- 最终成绩单 ---
准确率: 0.4786

详细分类报告:
              precision    recall  f1-score   support

       甲亢(1)       0.47      0.42      0.45        45
       甲减(2)       1.00      0.03      0.06        34
       正常(3)       0.47      0.77      0.59        61

    accuracy                           0.48       140
   macro avg       0.65      0.41      0.36       140
weighted avg       0.60      0.48      0.41       140


✅ TabPFN运行完成！


# 二分类策略：甲亢控制成功 vs 失败

## 医学逻辑
治疗目标：**让甲亢患者不再甲亢**
- ✅ **甲减(2)和正常(3)**：甲亢已控制，治疗成功
  - 甲减可以通过补充左旋甲状腺素轻松管理
  - 比甲亢更安全、更容易控制
- ❌ **甲亢(1)**：仍然甲亢，治疗失败
  - 需要二次治疗

## 二分类合并策略（新版本）
```
原始三分类:
  - 类别1 (甲亢) → 225例
  - 类别2 (甲减) → 168例  
  - 类别3 (正常) → 307例

合并后二分类:
  - 失败 (仍甲亢): 类别1 = 225例 (32.1%)
  - 成功 (已控制): 类别2 + 类别3 = 475例 (67.9%)
```

## 预期效果
- **样本更不平衡**（225 vs 475，但这是真实分布）
- **决策边界更明确**：甲亢 vs 非甲亢
- **可能准确率更高**：因为这是实际临床关注的问题


In [7]:
# ==========================================
# 二分类版本：Random Forest
# ==========================================
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_predict, RandomizedSearchCV
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

def get_features_binary(df):
    """特征工程 + 二分类标签"""
    # 构造工程特征（与三分类一致）
    df['Dose_Intensity'] = df['Dose_mCi'] / df['Thyroid_Weight']
    df['TRAb_Density'] = df['TRAb'] / df['Thyroid_Weight']
    df['Effective_Dose'] = df['Dose_mCi'] * (df['Iodine_Uptake_24h'] / 100)
    
    features = ['Age', 'Weight', 'Thyroid_Weight', 'Ultrasound_Score', 
                'FT3', 'TRAb', 'Iodine_Uptake_24h', 'Dose_mCi',
                'Dose_Intensity', 'TRAb_Density', 'Effective_Dose']
    
    X = df[features].fillna(df[features].median())
    
    # 二分类标签（新策略）：1=失败(0), 2/3=成功(1)
    # 只要不是甲亢(1)就算成功
    y_binary = (df['Outcome'] != 1).astype(int)
    
    return X, y_binary, features

def run_rf_binary(file_path, auto_tune=True):
    """运行二分类RF"""
    print("=" * 60)
    print("二分类 Random Forest")
    print("=" * 60)
    
    # 加载数据（复用load_data函数）
    df = load_data(file_path)
    X, y, feat_names = get_features_binary(df)
    
    print(f"\n样本总数: {len(X)}")
    print(f"失败 (仍甲亢): {sum(y==0)} ({sum(y==0)/len(y)*100:.1f}%)")
    print(f"成功 (已控制): {sum(y==1)} ({sum(y==1)/len(y)*100:.1f}%)")
    
    if auto_tune:
        print("\n🔧 开始自动调优...")
        param_distributions = {
            'n_estimators': [100, 200, 300, 500],
            'max_depth': [3, 5, 7, 10, 15, None],
            'min_samples_split': [2, 5, 10, 20],
            'min_samples_leaf': [1, 2, 4, 8],
            'max_features': ['sqrt', 'log2', 0.3, 0.5],
            'class_weight': ['balanced', 'balanced_subsample']
        }
        
        rf_base = RandomForestClassifier(random_state=42)
        random_search = RandomizedSearchCV(
            estimator=rf_base,
            param_distributions=param_distributions,
            n_iter=30,
            cv=5,
            scoring='accuracy',
            n_jobs=-1,
            random_state=42,
            verbose=0
        )
        
        random_search.fit(X, y)
        rf = random_search.best_estimator_
        print(f"✅ 最佳准确率: {random_search.best_score_:.4f}")
        print(f"最佳参数: {random_search.best_params_}")
    else:
        rf = RandomForestClassifier(
            n_estimators=200, 
            max_depth=5, 
            class_weight='balanced', 
            random_state=42
        )
    
    # 5折交叉验证
    print("\n正在进行 5折交叉验证...")
    y_pred = cross_val_predict(rf, X, y, cv=5)
    
    print("\n--- 最终成绩单 ---")
    print(classification_report(y, y_pred, target_names=['仍甲亢(失败)', '已控制(成功)']))
    
    # 混淆矩阵
    cm = confusion_matrix(y, y_pred)
    print("\n混淆矩阵:")
    print(f"              预测:失败  预测:成功")
    print(f"实际:失败      {cm[0,0]:>6}    {cm[0,1]:>6}")
    print(f"实际:成功      {cm[1,0]:>6}    {cm[1,1]:>6}")
    
    # 特征重要性
    rf.fit(X, y)
    importances = pd.DataFrame({
        'Feature': feat_names, 
        'Importance': rf.feature_importances_
    }).sort_values(by='Importance', ascending=False)
    
    print("\n--- Top 5 关键特征 ---")
    print(importances.head(5).to_string(index=False))
    
    return rf

# 运行二分类RF
if __name__ == "__main__":
    file_name = "700.xlsx"
    rf_binary_model = run_rf_binary(file_name, auto_tune=True)


二分类 Random Forest

样本总数: 700
失败 (仍甲亢): 225 (32.1%)
成功 (已控制): 475 (67.9%)

🔧 开始自动调优...
✅ 最佳准确率: 0.7057
最佳参数: {'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': 15, 'class_weight': 'balanced'}

正在进行 5折交叉验证...

--- 最终成绩单 ---
              precision    recall  f1-score   support

     仍甲亢(失败)       0.57      0.36      0.44       225
     已控制(成功)       0.74      0.87      0.80       475

    accuracy                           0.71       700
   macro avg       0.65      0.61      0.62       700
weighted avg       0.68      0.71      0.68       700


混淆矩阵:
              预测:失败  预测:成功
实际:失败          80       145
实际:成功          61       414

--- Top 5 关键特征 ---
       Feature  Importance
Thyroid_Weight    0.164983
Effective_Dose    0.127590
Dose_Intensity    0.113046
          TRAb    0.095952
  TRAb_Density    0.095215


In [ ]:
# ==========================================
# 二分类版本：TabPFN
# ==========================================
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.impute import SimpleImputer

def run_tabpfn_binary(file_path):
    """运行二分类TabPFN"""
    print("=" * 60)
    print("二分类 TabPFN")
    print("=" * 60)
    
    # 加载数据
    df = load_and_clean_data(file_path)
    
    # 特征工程（与RF一致）
    df['Dose_Intensity'] = df['Dose_mCi'] / df['Thyroid_Weight']
    df['TRAb_Density'] = df['TRAb'] / df['Thyroid_Weight']
    df['Effective_Dose'] = df['Dose_mCi'] * (df['Iodine_Uptake_24h'] / 100)
    
    feature_cols = [
        'Age', 'Weight', 'Thyroid_Weight', 'Ultrasound_Score', 
        'FT3', 'TRAb', 'Iodine_Uptake_24h', 'Dose_mCi',
        'Dose_Intensity', 'TRAb_Density', 'Effective_Dose'
    ]
    
    X = df[feature_cols]
    
    # 二分类标签（新策略）：1=失败(0), 2/3=成功(1)
    # 只要不是甲亢(1)就算成功
    y_binary = (df['Outcome'] != 1).astype(int)
    
    print(f"\n样本总数: {len(X)}")
    print(f"失败 (仍甲亢): {sum(y_binary==0)} ({sum(y_binary==0)/len(y_binary)*100:.1f}%)")
    print(f"成功 (已控制): {sum(y_binary==1)} ({sum(y_binary==1)/len(y_binary)*100:.1f}%)")
    
    # 填补缺失值
    imputer = SimpleImputer(strategy='median')
    X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)
    
    # 切分数据
    X_train, X_test, y_train, y_test = train_test_split(
        X_imputed, y_binary, test_size=0.2, random_state=42, stratify=y_binary
    )
    
    print(f"训练集: {len(X_train)}, 测试集: {len(X_test)}")
    
    try:
        from tabpfn import TabPFNClassifier
    except ImportError as e:
        print(f"❌ TabPFN导入失败: {e}")
        return None

    classifier = TabPFNClassifier(device='cpu')
    
    print("\n正在进行 In-Context Learning...")
    classifier.fit(X_train, y_train)
    y_pred = classifier.predict(X_test)
    
    # 评估
    print("\n--- 最终成绩单 ---")
    print(f"准确率: {accuracy_score(y_test, y_pred):.4f}")
    print("\n详细分类报告:")
    print(classification_report(y_test, y_pred, target_names=['仍甲亢(失败)', '已控制(成功)']))
    
    # 混淆矩阵
    cm = confusion_matrix(y_test, y_pred)
    print("\n混淆矩阵:")
    print(f"              预测:失败  预测:成功")
    print(f"实际:失败      {cm[0,0]:>6}    {cm[0,1]:>6}")
    print(f"实际:成功      {cm[1,0]:>6}    {cm[1,1]:>6}")
    
    return classifier

# 运行二分类TabPFN
if __name__ == "__main__":
    file_name = "700.xlsx"
    tabpfn_binary_model = run_tabpfn_binary(file_name)


二分类 TabPFN

样本总数: 700
失败 (仍甲亢): 225 (32.1%)
成功 (已控制): 475 (67.9%)
训练集: 560, 测试集: 140

正在进行 In-Context Learning...


/home/l1q/anaconda3/envs/med/lib/python3.10/site-packages/tabpfn/classifier.py:604: UserWarning: Running on CPU with more than 200 samples may be slow.
Consider using a GPU or the tabpfn-client API: https://github.com/PriorLabs/tabpfn-client
  check_cpu_warning(



--- 最终成绩单 ---
准确率: 0.7429

详细分类报告:
              precision    recall  f1-score   support

     仍甲亢(失败)       0.68      0.38      0.49        45
     已控制(成功)       0.76      0.92      0.83        95

    accuracy                           0.74       140
   macro avg       0.72      0.65      0.66       140
weighted avg       0.73      0.74      0.72       140


混淆矩阵:
              预测:失败  预测:成功
实际:失败          17        28
实际:成功           8        87


[W107 21:25:15.168525566 AllocatorConfig.cpp:28] Warning: PYTORCH_CUDA_ALLOC_CONF is deprecated, use PYTORCH_ALLOC_CONF instead (function operator())
[W107 21:25:16.308289176 AllocatorConfig.cpp:28] Warning: PYTORCH_CUDA_ALLOC_CONF is deprecated, use PYTORCH_ALLOC_CONF instead (function operator())
[W107 21:25:17.994651871 AllocatorConfig.cpp:28] Warning: PYTORCH_CUDA_ALLOC_CONF is deprecated, use PYTORCH_ALLOC_CONF instead (function operator())
[W107 21:25:18.734486664 AllocatorConfig.cpp:28] Warning: PYTORCH_CUDA_ALLOC_CONF is deprecated, use PYTORCH_ALLOC_CONF instead (function operator())
[W107 21:25:18.445279564 AllocatorConfig.cpp:28] Warning: PYTORCH_CUDA_ALLOC_CONF is deprecated, use PYTORCH_ALLOC_CONF instead (function operator())
[W107 21:25:19.124765296 AllocatorConfig.cpp:28] Warning: PYTORCH_CUDA_ALLOC_CONF is deprecated, use PYTORCH_ALLOC_CONF instead (function operator())
[W107 21:25:20.836720323 AllocatorConfig.cpp:28] Warning: PYTORCH_CUDA_ALLOC_CONF is deprecated, use